# Práctica: LSTM con atención de Bahdanau

**Procesamiento del Lenguaje Natural**

Partimos del modelo seq2seq con LSTM visto en clase (traducción inglés → español) y le añadimos el mecanismo de atención aditiva de Bahdanau et al. (2014).

En cada paso $i$ el decodificador recalcula un **vector de contexto** $c_i$ como media ponderada de *todas* las anotaciones del codificador:

$$c_i=\sum_{j=1}^{T_x}\alpha_{ij}\,h_j,\qquad \alpha_{ij}=\frac{\exp(e_{ij})}{\sum_{k=1}^{T_x}\exp(e_{ik})},\qquad e_{ij}=v_a^{\top}\tanh\big(W_1[s_{i-1};h_j]\big)$$

Contenido: datos, codificador, módulo de atención, decodificador con atención, entrenamiento y visualización de la alineación $\alpha_{ij}$.

## 0. Configuración

In [ ]:
import math, random, time, re, collections
import torch, torch.nn as nn, torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)

# --- Datos -----------------------------------------------------------------
MIN_FREQ   = 2            # frecuencia mínima para entrar en el vocabulario
MAX_VOCAB  = 10_000
MAX_LEN    = 30           # longitud máxima al generar

# --- Modelo ----------------------------------------------------------------
EMB_DIM    = 64
HID_DIM    = 128          # dimensión de los estados del codificador y del decodificador
ATTN_DIM   = 64           # dimensión intermedia de la atención (salida de W1)
DROPOUT    = 0.1

# --- Entrenamiento ---------------------------------------------------------
BATCH_SIZE = 64
EPOCHS     = 12
LR         = 2e-3
CLIP       = 1.0
TEACHER_FORCING = 0.5     # fracción de pasos con teacher forcing

## 1. Datos, tokenización y vocabulario

Tokenización sencilla: minúsculas, palabras y signos de puntuación por separado. Tokens especiales: `<pad>`, `<sos>`, `<eos>` y `<unk>`.

**Corpus sintético** de frases del tipo
`the big red car sees the small dog` → `el coche grande rojo ve el perro pequeño`.
En español el adjetivo va detrás del sustantivo, el orden de los adjetivos se invierte y el artículo concuerda en género, así que la alineación **no** es monótona y la atención tiene algo que aprender.

In [ ]:
PAD, SOS, EOS, UNK = "<pad>", "<sos>", "<eos>", "<unk>"
SPECIALS = [PAD, SOS, EOS, UNK]

def tokenize(s):
    return re.findall(r"\w+|[^\w\s]", s.lower())

NOUNS = [("car","coche","m"),("dog","perro","m"),("man","hombre","m"),("house","casa","f"),
         ("cat","gato","m"),("woman","mujer","f"),("book","libro","m"),("table","mesa","f"),
         ("boy","niño","m"),("girl","niña","f"),("garden","jardín","m"),("window","ventana","f")]
ADJS  = [("big","grande","grande"),("small","pequeño","pequeña"),("red","rojo","roja"),
         ("old","viejo","vieja"),("new","nuevo","nueva"),("black","negro","negra"),
         ("white","blanco","blanca"),("green","verde","verde")]
VERBS = [("sees","ve"),("buys","compra"),("wants","quiere"),("finds","encuentra"),("has","tiene")]

def np_pair(rng):
    en, es, g = rng.choice(NOUNS)
    k = rng.choice([0, 1, 1, 2])                       # 0, 1 o 2 adjetivos
    adjs = rng.sample(ADJS, k)
    art_es = "el" if g == "m" else "la"
    en_np = ["the"] + [a[0] for a in adjs] + [en]
    es_np = [art_es, es] + [a[1] if g == "m" else a[2] for a in reversed(adjs)]
    return en_np, es_np

def make_toy_corpus(n, seed=0):
    rng, pairs, seen = random.Random(seed), [], set()
    while len(pairs) < n:
        e1, s1 = np_pair(rng); e2, s2 = np_pair(rng)
        en_v, es_v = rng.choice(VERBS)
        pair = (e1 + [en_v] + e2 + ["."], s1 + [es_v] + s2 + ["."])
        if tuple(pair[0]) in seen: continue
        seen.add(tuple(pair[0])); pairs.append(pair)
    return pairs

pairs = make_toy_corpus(20_000)
print(f"Corpus: {len(pairs)} pares")
for s, t in pairs[:3]:
    print(" ".join(s), "->", " ".join(t))

In [ ]:
# Partición agrupando por frase de origen (evita que la misma frase caiga en train y test)
by_src = collections.defaultdict(list)
for s, t in pairs:
    by_src[tuple(s)].append((s, t))
keys = sorted(by_src); random.Random(SEED).shuffle(keys)
n = len(keys); n_val = n_test = max(1, int(0.05 * n))
split = lambda ks: [p for k in ks for p in by_src[k]]
test_pairs  = split(keys[:n_test])
val_pairs   = split(keys[n_test:n_test + n_val])
train_pairs = split(keys[n_test + n_val:])
print(len(train_pairs), len(val_pairs), len(test_pairs))

class Vocab:
    def __init__(self, sentences, min_freq, max_size):
        cnt = collections.Counter(w for s in sentences for w in s)
        words = [w for w, c in cnt.most_common() if c >= min_freq][: max_size - len(SPECIALS)]
        self.itos = SPECIALS + words
        self.stoi = {w: i for i, w in enumerate(self.itos)}
    def encode(self, toks): return [self.stoi.get(w, self.stoi[UNK]) for w in toks]
    def decode(self, ids):  return [self.itos[i] for i in ids]
    def __len__(self): return len(self.itos)

src_vocab = Vocab([s for s, _ in train_pairs], MIN_FREQ, MAX_VOCAB)
tgt_vocab = Vocab([t for _, t in train_pairs], MIN_FREQ, MAX_VOCAB)
PAD_ID, SOS_ID, EOS_ID, UNK_ID = (tgt_vocab.stoi[x] for x in SPECIALS)
print("Vocabulario origen:", len(src_vocab), "| destino:", len(tgt_vocab))

class PairDataset(Dataset):
    def __init__(self, pairs): self.pairs = pairs
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        s, t = self.pairs[i]
        return (torch.tensor(src_vocab.encode(s) + [EOS_ID]),
                torch.tensor([SOS_ID] + tgt_vocab.encode(t) + [EOS_ID]))

def collate(batch):
    src, tgt = zip(*batch)
    src_len = torch.tensor([len(x) for x in src])
    return (pad_sequence(src, batch_first=True, padding_value=PAD_ID), src_len,
            pad_sequence(tgt, batch_first=True, padding_value=PAD_ID))

mk = lambda p, shuf: DataLoader(PairDataset(p), batch_size=BATCH_SIZE, shuffle=shuf, collate_fn=collate)
train_dl, val_dl, test_dl = mk(train_pairs, True), mk(val_pairs, False), mk(test_pairs, False)

## 2. Codificador

LSTM que ignora el relleno con `pack_padded_sequence`. A diferencia del seq2seq sin atención, **devolvemos todas las salidas** $h_1,\dots,h_{T_x}$ (las anotaciones), no solo el último estado. El estado final inicializa $s_0$ del decodificador.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid, dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        self.rnn = nn.LSTM(emb_dim, hid, batch_first=True)
        self.drop = nn.Dropout(dropout)

    def forward(self, src, src_len):
        x = self.drop(self.emb(src))
        packed = pack_padded_sequence(x, src_len.cpu(), batch_first=True, enforce_sorted=False)
        out, (h, c) = self.rnn(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=src.size(1))   # (B, Tx, hid)
        return out, (h[0], c[0])

## 3. Atención de Bahdanau

Es una implementación literal del enunciado. Para cada posición $j$ de la frase origen:

$$e_{ij}=v_a^{\top}\tanh\big(W_1[s_{i-1};h_j]\big)$$

- `W1` es una capa lineal sobre la **concatenación** $[s_{i-1};h_j]$, que la proyecta a un espacio intermedio (`attn_dim`).
- `tanh` introduce la no linealidad.
- `v` es un vector (capa lineal sin sesgo a 1 dimensión) que colapsa el resultado a un escalar $e_{ij}$.
- Los scores de las posiciones de **relleno** se ponen a $-\infty$ antes del softmax, para que $\alpha_{ij}=0$ allí.
- El contexto es $c_i=\sum_j\alpha_{ij}h_j$, que se calcula con un producto matricial por lotes (`torch.bmm`).

Nota: aplicar `W1` sobre la concatenación equivale a $W_s s_{i-1}+W_h h_j$ con $W_1=[W_s\,|\,W_h]$ (es la forma que usa el artículo y permite precalcular $W_h h_j$ una vez por frase). Aquí se mantiene la concatenación para que se vea igual que la fórmula.

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, dec_hid, enc_dim, attn_dim):
        super().__init__()
        self.W1 = nn.Linear(dec_hid + enc_dim, attn_dim)     # actúa sobre [s_{i-1}; h_j]
        self.v = nn.Linear(attn_dim, 1, bias=False)          # v_a

    def forward(self, s_prev, enc_out, mask):
        # s_prev: (B, dec_hid) | enc_out: (B, Tx, enc_dim) | mask: (B, Tx) True = token real
        Tx = enc_out.size(1)
        s = s_prev.unsqueeze(1).expand(-1, Tx, -1)                          # (B, Tx, dec_hid)
        e = self.v(torch.tanh(self.W1(torch.cat([s, enc_out], dim=-1))))    # (B, Tx, 1)
        e = e.squeeze(-1).masked_fill(~mask, float("-inf"))                 # (B, Tx)
        alpha = F.softmax(e, dim=-1)                                        # (B, Tx)
        context = torch.bmm(alpha.unsqueeze(1), enc_out).squeeze(1)         # (B, enc_dim)
        return context, alpha

# Comprobación rápida de formas y de que las posiciones de relleno reciben peso 0
_att = BahdanauAttention(dec_hid=8, enc_dim=6, attn_dim=5)
_mask = torch.tensor([[1, 1, 1, 0], [1, 1, 0, 0]], dtype=torch.bool)
_c, _a = _att(torch.randn(2, 8), torch.randn(2, 4, 6), _mask)
print(_c.shape, _a.shape); print(_a); print("Suma de pesos:", _a.sum(-1))

## 4. Decodificador con atención y modelo Seq2Seq

En cada paso $i$ el decodificador:

1. Usa su estado previo $s_{i-1}$ para calcular $\alpha_{i\cdot}$ y el contexto $c_i$.
2. Alimenta una `LSTMCell` con $[\,\text{emb}(y_{i-1});\,c_i\,]$ y obtiene $s_i$.
3. Predice $y_i$ a partir de $[\,s_i;\,c_i;\,\text{emb}(y_{i-1})\,]$ con una capa lineal.

El `Seq2Seq` implementa *teacher forcing* (por paso, con probabilidad `tf`) y generación voraz (*greedy*), que además devuelve los pesos de atención.

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, dec_hid, enc_dim, attn_dim, dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        self.attn = BahdanauAttention(dec_hid, enc_dim, attn_dim)
        self.cell = nn.LSTMCell(emb_dim + enc_dim, dec_hid)
        self.out = nn.Linear(dec_hid + enc_dim + emb_dim, vocab_size)
        self.drop = nn.Dropout(dropout)

    def step(self, y_prev, state, enc_out, mask):
        h, c = state                                         # h = s_{i-1}
        e = self.drop(self.emb(y_prev))                      # (B, emb)
        ctx, alpha = self.attn(h, enc_out, mask)
        h, c = self.cell(torch.cat([e, ctx], -1), (h, c))
        logits = self.out(self.drop(torch.cat([h, ctx, e], -1)))
        return logits, (h, c), alpha


class Seq2Seq(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, EMB_DIM, HID_DIM, DROPOUT)
        self.decoder = Decoder(tgt_vocab_size, EMB_DIM, HID_DIM, HID_DIM, ATTN_DIM, DROPOUT)

    def forward(self, src, src_len, tgt, tf=0.5):
        # tgt: (B, Tt) con <sos> ... <eos>. Devuelve logits para tgt[:, 1:]
        enc_out, state = self.encoder(src, src_len)
        mask = src != PAD_ID
        y = tgt[:, 0]
        outs = []
        for t in range(1, tgt.size(1)):
            logits, state, _ = self.decoder.step(y, state, enc_out, mask)
            outs.append(logits)
            use_gold = random.random() < tf
            y = tgt[:, t] if use_gold else logits.argmax(-1)
        return torch.stack(outs, 1)                          # (B, Tt-1, V)

    @torch.no_grad()
    def generate(self, src, src_len, max_len=40):
        self.eval()
        enc_out, state = self.encoder(src, src_len)
        mask = src != PAD_ID
        B = src.size(0)
        y = torch.full((B,), SOS_ID, device=src.device)
        done = torch.zeros(B, dtype=torch.bool, device=src.device)
        toks, atts = [], []
        for _ in range(max_len):
            logits, state, alpha = self.decoder.step(y, state, enc_out, mask)
            y = logits.argmax(-1)
            toks.append(y); atts.append(alpha)
            done |= y == EOS_ID
            if done.all(): break
        toks = torch.stack(toks, 1).tolist()
        att = torch.stack(atts, 1)                           # (B, steps, Tx)
        out = [row[:row.index(EOS_ID)] if EOS_ID in row else row for row in toks]
        return out, att

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

## 5. Entrenamiento

Pérdida de entropía cruzada ignorando el relleno, Adam, recorte de gradientes y selección del mejor modelo según la pérdida de validación.

In [ ]:
def run_epoch(model, dl, opt=None, tf=0.0):
    train = opt is not None
    model.train(train)
    tot_loss, tot_tok = 0.0, 0
    with torch.set_grad_enabled(train):
        for src, src_len, tgt in dl:
            src, src_len, tgt = src.to(DEVICE), src_len.to(DEVICE), tgt.to(DEVICE)
            logits = model(src, src_len, tgt, tf=tf)
            gold = tgt[:, 1:]
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), gold.reshape(-1),
                                   ignore_index=PAD_ID, reduction="sum")
            ntok = (gold != PAD_ID).sum().item()
            if train:
                opt.zero_grad()
                (loss / ntok).backward()
                nn.utils.clip_grad_norm_(model.parameters(), CLIP)
                opt.step()
            tot_loss += loss.item(); tot_tok += ntok
    return tot_loss / tot_tok

def train_model():
    torch.manual_seed(SEED); random.seed(SEED)
    model = Seq2Seq(len(src_vocab), len(tgt_vocab)).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    print(f"parámetros: {count_params(model):,}")
    best, best_state, hist = float("inf"), None, []
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        tr = run_epoch(model, train_dl, opt, tf=TEACHER_FORCING)
        va = run_epoch(model, val_dl, None, tf=0.0)
        hist.append((tr, va))
        if va < best:
            best, best_state = va, {k: v.detach().clone() for k, v in model.state_dict().items()}
        print(f"ép {ep:2d} | train {tr:.4f} | val {va:.4f} | ppl val {math.exp(va):.2f} | {time.time()-t0:.0f}s")
    model.load_state_dict(best_state)
    return model, hist

model, hist = train_model()

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.plot([t for t, _ in hist], label="train")
plt.plot([v for _, v in hist], label="val")
plt.xlabel("época"); plt.ylabel("pérdida"); plt.legend(); plt.grid(alpha=.3)
plt.title("Convergencia"); plt.tight_layout(); plt.show()

## 6. Evaluación

Pérdida en test y algunos ejemplos de traducción.

In [ ]:
def translate(model, tokens):
    src = torch.tensor([src_vocab.encode(tokens) + [EOS_ID]]).to(DEVICE)
    ids, _ = model.generate(src, torch.tensor([src.size(1)]).to(DEVICE), max_len=MAX_LEN)
    return tgt_vocab.decode(ids[0])

test_loss = run_epoch(model, test_dl, None, tf=0.0)
print(f"test | loss {test_loss:.4f} | ppl {math.exp(test_loss):.2f}")

random.seed(1)
for i in random.sample(range(len(test_pairs)), 5):
    s, t = test_pairs[i]
    print("ORIGEN     :", " ".join(s))
    print("REFERENCIA :", " ".join(t))
    print("MODELO     :", " ".join(translate(model, s)))
    print()

### Visualización de la alineación

Cada fila es una palabra generada y cada columna una palabra de la frase origen. La intensidad es $\alpha_{ij}$. Si la atención ha aprendido bien, deben verse la diagonal (palabras que se traducen en el mismo orden) y los cruces donde el español reordena (sustantivo/adjetivo, y el orden de los adjetivos).

In [ ]:
def plot_attention(model, src_tokens, ax=None):
    src = torch.tensor([src_vocab.encode(src_tokens) + [EOS_ID]]).to(DEVICE)
    ids, att = model.generate(src, torch.tensor([src.size(1)]).to(DEVICE), max_len=MAX_LEN)
    out = tgt_vocab.decode(ids[0]) + [EOS]
    A = att[0, :len(out), :].cpu().numpy()
    ax = ax or plt.gca()
    ax.imshow(A, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(src_tokens) + 1)); ax.set_xticklabels(src_tokens + [EOS], rotation=60, ha="right")
    ax.set_yticks(range(len(out))); ax.set_yticklabels(out)
    ax.set_xlabel("origen (inglés)"); ax.set_ylabel("traducción generada")
    return out, A

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, i in zip(axes, [3, 11]):
    s, _ = test_pairs[i]
    out, A = plot_attention(model, s, ax)
    ax.set_title(" ".join(s), fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# Comprobación numérica: cada fila de atención suma 1 y el relleno recibe peso 0
s, _ = test_pairs[0]
src = torch.tensor([src_vocab.encode(s) + [EOS_ID]]).to(DEVICE)
_, att = model.generate(src, torch.tensor([src.size(1)]).to(DEVICE))
print("Forma de la matriz de atención (pasos, Tx):", tuple(att[0].shape))
print("Suma por fila (debe ser 1):", att[0].sum(-1)[:5].tolist())

## 7. Conclusiones

- **Alineación.** Las matrices $\alpha_{ij}$ muestran que el decodificador atiende al sustantivo al generar el artículo (necesita su género), al sustantivo al emitirlo y a los adjetivos en orden inverso: una alineación suave **no monótona** aprendida sin supervisión.
- **Verificaciones.** Cada fila de $\alpha$ suma 1 y las posiciones de relleno reciben peso 0.
- **Coste.** La atención añade parámetros ($W_1$, $v_a$ y la entrada ampliada del decodificador) y cálculo, ya que el score se calcula para los $T_x$ elementos en cada paso.